In [4]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
from datetime import datetime
from working_data import clean_five_minute_data, add_partial_hour_ohlc, add_timing, normalize_by_window, clean_hour_data, split_multiresolution_chunks, regression_label_df, normalize_partial_hour
from constants.global_constants import *
from modeler import create_regression_model

In [5]:
try:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f"Mixed precision policy set: {policy.name}")
    
    # Verify it's working
    print(f"Compute dtype: {policy.compute_dtype}")  # Should be float16
    print(f"Variable dtype: {policy.variable_dtype}")  # Should be float32
except Exception as e:
    print(f"Could not enable mixed precision: {e}")

Mixed precision policy set: mixed_float16
Compute dtype: float16
Variable dtype: float32


In [6]:
starting_dir = "data/final_data"
working_path = "data/regression"
instruments = os.listdir(starting_dir)

In [5]:
# for instrument in instruments:
#     df = pd.read_csv(f"{starting_dir}/{instrument}/five_minutes.csv")
#     print(f"Processing {instrument}...")
#     print(f"  5 minute Original data: {len(df)} rows")
#     print(f"  5 minute Time range: {datetime.fromtimestamp(df['time'].min())} to {datetime.fromtimestamp(df['time'].max())}")
#     df = clean_five_minute_data(df)
#     print(f"  Cleaned data: {len(df)} rows")
#     df = add_timing(df)
#     df = add_partial_hour_ohlc(df)
#     df = normalize_by_window(
#         df, 
#         window_size=NORMALIZING_WINDOW_SIZE, 
#         low_col='low',
#         high_col='high',
#         normalizing_cols=[
#             'open',
#             'high',
#             'low',
#             'close'
#         ],
#         label_cols=['open', 'close'])

#     hour_df = pd.read_csv(f"{starting_dir}/{instrument}/hours.csv")
#     print(f"  Hour Original data: {len(hour_df)} rows")
#     print(f"  Hour Time range: {datetime.fromtimestamp(hour_df['time'].min())} to {datetime.fromtimestamp(hour_df['time'].max())}")
#     hour_df = clean_hour_data(hour_df)
#     print(f"  Cleaned data: {len(hour_df)} rows")
#     hour_df = add_timing(hour_df)
#     hour_df = normalize_by_window(
#         hour_df, 
#         window_size=NORMALIZING_WINDOW_SIZE, 
#         low_col='low',
#         high_col='high',
#         normalizing_cols=[
#             'open',
#             'high',
#             'low',
#             'close'
#         ],
#         label_cols=['open', 'close'],
#         add_partial_hour=True)

#     df = normalize_partial_hour(df, hour_df)

#     print(f"Labeling...\n\n\n")
#     df = regression_label_df(df, window_size=REGRESSION_LABELING_WINDOW_SIZE, 
#                     positive_slope=POSITIVE_SLOPE, 
#                     negative_slope=NEGATIVE_SLOPE,
#                     starting_hour=9,
#                     ending_hour=18,
#                     lookback_window=LABEL_LOOKBACK)


#     os.makedirs(f"{working_path}/{instrument}", exist_ok=True)

#     hour_df.to_csv(f"{working_path}/{instrument}/hour.csv", index=False)
#     split_multiresolution_chunks(df_5min=df,
#                                 df_hour=hour_df,
#                                 dump_path=f"{working_path}/{instrument}",
#                                 chunk_size=20000,
#                                 hour_lookback=OTHER_TOKENS,
#                                 lookback=NUM_TOKENS,
#                                 cols=[
#                                     'time',
#                                     'position_in_hour',
#                                     'partial_hour_length',
#                                     'open_normalized',
#                                     'high_normalized',
#                                     'low_normalized',
#                                     'close_normalized',
#                                     'partial_open_normalized',
#                                     'partial_high_normalized',
#                                     'partial_low_normalized',
#                                     'partial_close_normalized',
#                                     'include',
#                                     'target_high',
#                                     'target_low'
#                                 ])

In [7]:
import os
from generators.regression_multi_instrument_data_generator import InstrumentConfig, MultiInstrumentDatasetConfig, create_multi_instrument_dataset
from constants.global_constants import FEATURES, NUM_TOKENS, OTHER_TOKENS, BATCH_SIZE, LOOKBACK_WINDOW


instruments = os.listdir(working_path)
feature_cols = FEATURES

def get_datasets_and_steps(instruments=instruments, working_path=working_path, feature_cols=feature_cols):
    train_instrument_configs = []
    val_instrument_configs = []
    test_instrument_configs = []

    for instrument in instruments:
        train_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/training"
            )
        )
        val_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/validation"
            )
        )
        test_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/testing"
            )
        )

    train_config = MultiInstrumentDatasetConfig(
        instruments=train_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=True,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )

    val_config = MultiInstrumentDatasetConfig(
        instruments=val_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=False,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )

    test_config = MultiInstrumentDatasetConfig(
        instruments=test_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=False,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )


    train_dataset, train_rows = create_multi_instrument_dataset(
        config=train_config,
        repeat_dataset=True
    )
    val_dataset, val_rows = create_multi_instrument_dataset(
        config=val_config,
        repeat_dataset=True
    )
    test_dataset, test_rows = create_multi_instrument_dataset(
        config=test_config,
        repeat_dataset=True
    )

    train_steps = train_rows//BATCH_SIZE
    val_steps = val_rows//BATCH_SIZE
    test_steps = test_rows//BATCH_SIZE

    return (
        (train_dataset, val_dataset, test_dataset),
        (train_steps, val_steps, test_steps)
    )

In [10]:
(train_dataset, val_dataset, test_dataset), (train_steps, val_steps, test_steps) = get_datasets_and_steps()

2025-07-30 09:26:02,733 - INFO - Discovered 36 chunk files for GOLD#
2025-07-30 09:26:02,734 - INFO - Loading hourly data for GOLD#...
2025-07-30 09:26:02,771 - INFO - Loaded hourly data for GOLD#: 64398 rows
2025-07-30 09:26:02,773 - INFO - Applied time threshold for GOLD#: 1369198800
2025-07-30 09:26:02,774 - INFO - Building indices for GOLD#...
2025-07-30 09:26:04,136 - INFO - Built indices for GOLD#: 38017 valid samples
2025-07-30 09:26:04,137 - INFO - Discovered 85 chunk files for USDCAD#
2025-07-30 09:26:04,138 - INFO - Loading hourly data for USDCAD#...
2025-07-30 09:26:04,145 - INFO - Loaded hourly data for USDCAD#: 143877 rows
2025-07-30 09:26:04,146 - INFO - Applied time threshold for USDCAD#: 979318800
2025-07-30 09:26:04,147 - INFO - Building indices for USDCAD#...
2025-07-30 09:26:07,316 - INFO - Built indices for USDCAD#: 80657 valid samples
2025-07-30 09:26:07,317 - INFO - Discovered 84 chunk files for CHFJPY#
2025-07-30 09:26:07,318 - INFO - Loading hourly data for CHFJ

In [8]:
def get_naive_baseline_metrics(val_dataset, val_steps):
    """
    Calculate naive baseline metrics for target_high predictions.
    Uses mean prediction as the naive baseline.
    
    Args:
        val_dataset: TensorFlow dataset from create_multi_instrument_dataset
        val_steps: Number of validation steps/batches to process
        
    Returns:
        dict: Contains baseline_value, mae, mse, rmse
    """
    # Collect all target_high values
    all_target_highs = []
    
    for i, batch in enumerate(val_dataset):
        if i >= val_steps:
            break
        (main_input, hourly_input, partial, position, hourly_position), targets = batch
        target_highs = targets['target_high'].numpy()
        all_target_highs.extend(target_highs)
    
    all_target_highs = np.array(all_target_highs)
    
    # Calculate baseline (mean of all targets)
    baseline_value = np.mean(all_target_highs)
    
    # Create predictions (always predict the mean)
    predictions = np.full_like(all_target_highs, baseline_value)
    
    # Calculate metrics
    mae = np.mean(np.abs(predictions - all_target_highs))
    mse = np.mean((predictions - all_target_highs) ** 2)
    rmse = np.sqrt(mse)
    
    return {
        'baseline_value': baseline_value,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'total_samples': len(all_target_highs)
    }

In [9]:
get_naive_baseline_metrics(val_dataset, val_steps)

{'baseline_value': 3.4212673,
 'mae': 2.8617032,
 'mse': 17.394136,
 'rmse': 4.1706276,
 'total_samples': 244032}

In [8]:
from regression_losses import asymmetric_huber_loss_single, profit_accuracy_metric, profit_precision_metric, profit_recall_metric

def compile_model_lightweight(model, updelta, downdelta):
    """
    Streamlined compilation with only the most important metrics
    Mixed precision compatible.
    """
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
        loss={
            'target_high': asymmetric_huber_loss_single(
                delta=1.5, 
                underestimate_weight=2.2, 
                overestimate_weight=0.6
            ),
            'target_low': asymmetric_huber_loss_single(
                delta=1.8,
                underestimate_weight=0.8,
                overestimate_weight=2.5
            )
        },
        loss_weights={
            'target_high': 1.3,
            'target_low': 1.0
        },
        metrics={
            'target_high': [
                'mae',
                'mse',
                profit_precision_metric(threshold=6.0),
                profit_recall_metric(threshold=6.0),
            ],
            'target_low': [
                'mae',
            ]
        }
    )
    return model


model = create_regression_model(feature_cols=feature_cols, d_model=R_D_MODEL, num_heads=R_NUM_HEADS, ff_dim=R_FF_DIM,
                                num_tokens=NUM_TOKENS, other_tokens=OTHER_TOKENS, training=False)
model.load_weights('models/regressor.keras')
model = compile_model_lightweight(model=model, updelta=6.0, downdelta=-1.0)


2025-07-30 09:25:45.288656: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-30 09:25:45.318422: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-30 09:25:45.318474: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-30 09:25:45.322870: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-07-30 09:25:45.322923: I external/local_xla/xla/stream_executor

In [11]:
model.evaluate(test_dataset, steps=test_steps)

I0000 00:00:1753860435.134717  338025 service.cc:145] XLA service 0x7f5fd80207a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753860435.134826  338025 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2025-07-30 09:27:15.261975: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1753860435.269918  338025 random_ops.cc:59] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. functional_1/stochastic_gated_transformer_block_1/random_uniform/RandomUniform
2025-07-30 09:27:15.534995: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1753860436.639250  338432 asm_compiler.cc:369] ptxas warning : Registers are

   1/3825 ━━━━━━━━━━━━━━━━━━━━ 11:41:04 11s/step - loss: 15.6978 - target_high_loss: 7.0142 - target_high_mae: 4.8633 - target_high_metric: 0.2162 - target_high_metric_1: 1.0000 - target_high_mse: 29.9092 - target_low_loss: 8.6837 - target_low_mae: 3.7129

I0000 00:00:1753860444.105675  338025 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3825/3825 ━━━━━━━━━━━━━━━━━━━━ 491s 126ms/step - loss: 10.6697 - target_high_loss: 5.5938 - target_high_mae: 3.7871 - target_high_metric: 0.2722 - target_high_metric_1: 0.5739 - target_high_mse: 21.8022 - target_low_loss: 5.0759 - target_low_mae: 2.6507


[10.254651069641113,
 5.420551300048828,
 4.834111213684082,
 3.716089963912964,
 0.2646436095237732,
 0.5510988831520081,
 21.090892791748047,
 2.5879435539245605]

In [9]:
model = create_regression_model(feature_cols=feature_cols, d_model=R_D_MODEL, num_heads=R_NUM_HEADS, ff_dim=R_FF_DIM,
                                num_tokens=NUM_TOKENS, other_tokens=OTHER_TOKENS)
model = compile_model_lightweight(model=model, updelta=6.0, downdelta=-1.0)
print(model.metrics_names)

['loss', 'compile_metrics']


In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


early_stopping = EarlyStopping(monitor='val_target_high_metric_1', 
                               patience=10,
                               mode='max', 
                               verbose=1)

model_checkpoint = ModelCheckpoint('models/regressor.keras', 
                                   monitor='val_target_high_metric_1', 
                                   save_best_only=True, 
                                   mode='max', 
                                   verbose=1)

history = model.fit(
    train_dataset,
    epochs=50,
    steps_per_epoch=train_steps,
    validation_data=val_dataset,
    validation_steps=val_steps,
    callbacks=[early_stopping, model_checkpoint]
)

Epoch 1/50


I0000 00:00:1753586167.807286    1692 service.cc:145] XLA service 0x7fe81c014e40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753586167.807408    1692 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2025-07-27 05:16:08.092368: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1753586168.109308    1692 random_ops.cc:59] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. functional_1/stochastic_gated_transformer_block_1/random_uniform/RandomUniform
2025-07-27 05:16:10.981354: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1753586186.910401    3051 asm_compiler.cc:369] ptxas warning : Registers are

18559/18559 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - loss: 14.9849 - target_high_loss: 11.9524 - target_high_mae: 3.9214 - target_high_metric: 0.1950 - target_high_metric_1: 0.2384 - target_high_mse: 27.0725 - target_low_loss: 3.0324 - target_low_mae: 5.8577 - target_low_metric: 0.4662

I0000 00:00:1753590807.499877   16527 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_42', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1753590807.934838   16532 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 168 bytes spill stores, 168 bytes spill loads

I0000 00:00:1753590808.189602   16529 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_2863', 24 bytes spill stores, 24 bytes spill loads

I0000 00:00:1753590808.535853   16532 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1753590808.549741   16531 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 20 bytes spill stores, 20 bytes spill loads

I0000 00:00:1753590808.813682   16517 


Epoch 1: val_target_high_metric_1 improved from -inf to 0.10945, saving model to models/regressor.keras
18559/18559 ━━━━━━━━━━━━━━━━━━━━ 5098s 272ms/step - loss: 14.9848 - target_high_loss: 11.9523 - target_high_mae: 3.9214 - target_high_metric: 0.1950 - target_high_metric_1: 0.2384 - target_high_mse: 27.0721 - target_low_loss: 3.0324 - target_low_mae: 5.8577 - target_low_metric: 0.4662 - val_loss: 11.5231 - val_target_high_loss: 9.5775 - val_target_high_mae: 2.7496 - val_target_high_metric: 0.2224 - val_target_high_metric_1: 0.1095 - val_target_high_mse: 16.1201 - val_target_low_loss: 1.9456 - val_target_low_mae: 3.9710 - val_target_low_metric: 0.4631
Epoch 2/50
18559/18559 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step - loss: 10.9020 - target_high_loss: 9.0411 - target_high_mae: 3.1759 - target_high_metric: 0.3346 - target_high_metric_1: 0.2985 - target_high_mse: 17.7924 - target_low_loss: 1.8609 - target_low_mae: 5.1656 - target_low_metric: 0.4600

I0000 00:00:1753595831.801893   30215 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_42', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1753595832.807677   30210 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_42', 4 bytes spill stores, 4 bytes spill loads




Epoch 2: val_target_high_metric_1 improved from 0.10945 to 0.31748, saving model to models/regressor.keras
18559/18559 ━━━━━━━━━━━━━━━━━━━━ 5030s 271ms/step - loss: 10.9020 - target_high_loss: 9.0411 - target_high_mae: 3.1759 - target_high_metric: 0.3346 - target_high_metric_1: 0.2985 - target_high_mse: 17.7924 - target_low_loss: 1.8609 - target_low_mae: 5.1656 - target_low_metric: 0.4600 - val_loss: 10.7520 - val_target_high_loss: 8.9306 - val_target_high_mae: 3.1304 - val_target_high_metric: 0.2914 - val_target_high_metric_1: 0.3175 - val_target_high_mse: 17.6185 - val_target_low_loss: 1.8210 - val_target_low_mae: 4.4545 - val_target_low_metric: 0.4632
Epoch 3/50
18559/18559 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - loss: 10.6803 - target_high_loss: 8.8626 - target_high_mae: 3.1534 - target_high_metric: 0.3501 - target_high_metric_1: 0.3219 - target_high_mse: 17.6075 - target_low_loss: 1.8178 - target_low_mae: 5.1149 - target_low_metric: 0.4603
Epoch 3: val_target_high_metric_1 did not i